In [23]:
import os
import argparse
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [2]:
def load_data(path):
    df = pd.read_csv(path, sep='|')
    if 'TransactionMonth' in df.columns:
        df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'], errors='coerce')
    # create derived cols
    df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
    df['Margin'] = df['TotalPremium'] - df['TotalClaims']
    return df

In [12]:
# Load dataset first
df = load_data("../data/MachineLearningRating_v3.txt")

# Create the claim indicator
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

# Create margin
df['Margin'] = df['TotalPremium'] - df['TotalClaims']


C:\Users\mubar\AppData\Local\Temp\ipykernel_2468\2885341563.py:2: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep='|')


In [3]:
def freq_by_group(df, group_col):
    tab = pd.crosstab(df[group_col], df['HasClaim'])
    freq = tab.div(tab.sum(axis=1), axis=0)[1]  # proportion with claim
    return tab, freq

In [15]:
def chi2_test(df, group_col):
    tab = pd.crosstab(df[group_col], df['HasClaim'])
    chi2, p, dof, expected = stats.chi2_contingency(tab)
    return chi2, p, dof, expected

def two_proportion_ztest(count, nobs):
    # count: array-like [successes_group1, successes_group2]
    stat, pval = sm.stats.proportions_ztest(count, nobs)
    return stat, pval

def anova_test(groups):
    f, p = stats.f_oneway(*groups)
    return f, p

def kruskal_test(groups):
    h, p = stats.kruskal(*groups)
    return h, p

def run_tests(df, outdir):
    os.makedirs(outdir, exist_ok=True)

    results = {}


 H0 1: Province risk differences

In [17]:
results = {}
print("Testing H0: No risk differences across provinces (frequency)")
chi2, p, dof, _ = chi2_test(df, 'Province')
results['province_freq_chi2'] = dict(chi2=chi2, p=p, dof=dof)
print(f"Chi2={chi2:.3f}, p={p:.6f}, dof={dof}")

Testing H0: No risk differences across provinces (frequency)
Chi2=104.191, p=0.000000, dof=8


frequency by province table

In [22]:
import os

outdir = "../reports/task3"
os.makedirs(outdir, exist_ok=True)

tab, freq = freq_by_group(df, 'Province')
freq.to_csv(os.path.join(outdir, 'province_claim_frequency.csv'))


In [25]:
claims = df[df['HasClaim'] == 1]
groups = [g['TotalClaims'].values for n,g in claims.groupby('Province')]

In [28]:
 # require at least 2 groups with sample size > 5
if len(groups) >= 2:
        # use Kruskal-Wallis non-parametric (safer)
        h, p = kruskal_test(groups)
        results['province_severity_kruskal'] = dict(H=h, p=p)
        print(f"Kruskal H={h:.3f}, p={p:.6f}")
else:
        print("Not enough provinces with claims to run severity test.")

Kruskal H=106.093, p=0.000000


In [29]:
 # ---------- H0 2: Zip code risk differences ----------
    # Focus on top-N postal codes by count to avoid tiny groups
top_n = 10
top_zips = df['PostalCode'].value_counts().nlargest(top_n).index.tolist()
df_topz = df[df['PostalCode'].isin(top_zips)]
print(f"Testing top {top_n} PostalCodes:", top_zips)

Testing top 10 PostalCodes: [2000, 122, 7784, 299, 7405, 458, 8000, 2196, 470, 7100]


In [31]:
chi2, p, dof, _ = chi2_test(df_topz, 'PostalCode')
results['zip_freq_chi2_top10'] = dict(chi2=chi2, p=p, dof=dof)
print(f"Zip Chi2 (top10)={chi2:.3f}, p={p:.6f}")

Zip Chi2 (top10)=72.649, p=0.000000


In [32]:
 # severity across zipcodes (top10)
claims_topz = df_topz[df_topz['HasClaim'] == 1]
groups = [g['TotalClaims'].values for n,g in claims_topz.groupby('PostalCode')]
if len(groups) >= 2:
        h, p = kruskal_test(groups)
        results['zip_severity_kruskal_top10'] = dict(H=h, p=p)
        print(f"Zip Kruskal H (top10)={h:.3f}, p={p:.6f}")
else:
        print("Not enough zip groups with claims for severity test.")

Zip Kruskal H (top10)=41.428, p=0.000004


In [33]:
# ---------- H0 3: Margin difference between zip codes ----------
    # Use margin (continuous) and ANOVA/Kruskal on top zipcodes
groups_margin = [g['Margin'].values for n,g in df_topz.groupby('PostalCode')]
if len(groups_margin) >= 2:
        # check normality quickly with Shapiro for small groups is expensive; use Kruskal
        h, p = kruskal_test(groups_margin)
        results['zip_margin_kruskal_top10'] = dict(H=h, p=p)
        print(f"Zip Margin Kruskal (top10) H={h:.3f}, p={p:.6f}")
else:
        print("Not enough zip groups for margin test.")

Zip Margin Kruskal (top10) H=4931.140, p=0.000000


In [34]:
 # ---------- H0 4: Gender differences ----------
    # Frequency test (two-proportion z-test)
print("Testing gender differences (frequency)")
gender_tab = pd.crosstab(df['Gender'], df['HasClaim'])
gender_tab.to_csv(os.path.join(outdir, 'gender_claim_table.csv'))

Testing gender differences (frequency)


In [35]:
# Ensure exactly two genders captured; if more, pick two main categories (Male/Female)
if set(['M','F']).issubset(set(df['Gender'].unique())):
        # successes and nobs
        success = [gender_tab.loc['M',1], gender_tab.loc['F',1]]
        nobs = [gender_tab.loc['M',:].sum(), gender_tab.loc['F',:].sum()]
        stat, pval = two_proportion_ztest(success, nobs)
        results['gender_freq_ztest'] = dict(z=stat, p=pval, success=success, nobs=nobs)
        print(f"Gender z={stat:.3f}, p={pval:.6f}")
else:
        # fallback: chi2 across all genders captured
        chi2, p, dof, _ = chi2_test(df, 'Gender')
        results['gender_freq_chi2'] = dict(chi2=chi2, p=p, dof=dof)
        print(f"Gender Chi2={chi2:.3f}, p={p:.6f}")

Gender Chi2=7.256, p=0.026570


In [36]:
 # Severity by gender (claims only)
claims_gender = claims.groupby('Gender')['TotalClaims'].agg(['count','mean','std'])
claims_gender.to_csv(os.path.join(outdir, 'claims_by_gender.csv'))

In [37]:
# If two main genders, perform t-test or Mann-Whitney
genders_present = [g for g in df['Gender'].unique() if g in claims['Gender'].unique()]
if len(genders_present) >= 2:
        grp_vals = [claims[claims['Gender'] == g]['TotalClaims'].values for g in genders_present]
        # use Mann-Whitney U (nonparametric)
        if len(grp_vals[0])>10 and len(grp_vals[1])>10:
            u, p = stats.mannwhitneyu(grp_vals[0], grp_vals[1], alternative='two-sided')
            results['gender_severity_mwu'] = dict(U=u, p=p, genders=genders_present)
            print(f"Mann-Whitney U={u:.3f}, p={p:.6f}")
        else:
            print("Insufficient sample size for robust gender severity test.")
else:
        print("Not enough gender groups with claims to compare severity.")

Mann-Whitney U=138381.500, p=0.084725


In [48]:
def run_tests(df, outdir):
    os.makedirs(outdir, exist_ok=True)

    results = {}

    # ---------- H0 1: Province risk differences ----------
    print("Testing H0: No risk differences across provinces (frequency)")
    chi2, p, dof, _ = chi2_test(df, 'Province')
    results['province_freq_chi2'] = dict(chi2=chi2, p=p, dof=dof)
    print(f"Chi2={chi2:.3f}, p={p:.6f}, dof={dof}")

    # frequency by province table
    tab, freq = freq_by_group(df, 'Province')
    freq.to_csv(os.path.join(outdir, 'province_claim_frequency.csv'))

    # severity by province (only claims > 0)
    claims = df[df['HasClaim'] == 1]
    groups = [g['TotalClaims'].values for n,g in claims.groupby('Province')]
    # require at least 2 groups with sample size > 5
    if len(groups) >= 2:
        # use Kruskal-Wallis non-parametric (safer)
        h, p = kruskal_test(groups)
        results['province_severity_kruskal'] = dict(H=h, p=p)
        print(f"Kruskal H={h:.3f}, p={p:.6f}")
    else:
        print("Not enough provinces with claims to run severity test.")

    # ---------- H0 2: Zip code risk differences ----------
    # Focus on top-N postal codes by count to avoid tiny groups
    top_n = 10
    top_zips = df['PostalCode'].value_counts().nlargest(top_n).index.tolist()
    df_topz = df[df['PostalCode'].isin(top_zips)]
    print(f"Testing top {top_n} PostalCodes:", top_zips)

    chi2, p, dof, _ = chi2_test(df_topz, 'PostalCode')
    results['zip_freq_chi2_top10'] = dict(chi2=chi2, p=p, dof=dof)
    print(f"Zip Chi2 (top10)={chi2:.3f}, p={p:.6f}")

    # severity across zipcodes (top10)
    claims_topz = df_topz[df_topz['HasClaim'] == 1]
    groups = [g['TotalClaims'].values for n,g in claims_topz.groupby('PostalCode')]
    if len(groups) >= 2:
        h, p = kruskal_test(groups)
        results['zip_severity_kruskal_top10'] = dict(H=h, p=p)
        print(f"Zip Kruskal H (top10)={h:.3f}, p={p:.6f}")
    else:
        print("Not enough zip groups with claims for severity test.")

    # ---------- H0 3: Margin difference between zip codes ----------
    # Use margin (continuous) and ANOVA/Kruskal on top zipcodes
    groups_margin = [g['Margin'].values for n,g in df_topz.groupby('PostalCode')]
    if len(groups_margin) >= 2:
        # check normality quickly with Shapiro for small groups is expensive; use Kruskal
        h, p = kruskal_test(groups_margin)
        results['zip_margin_kruskal_top10'] = dict(H=h, p=p)
        print(f"Zip Margin Kruskal (top10) H={h:.3f}, p={p:.6f}")
    else:
        print("Not enough zip groups for margin test.")

    # ---------- H0 4: Gender differences ----------
    # Frequency test (two-proportion z-test)
    print("Testing gender differences (frequency)")
    gender_tab = pd.crosstab(df['Gender'], df['HasClaim'])
    gender_tab.to_csv(os.path.join(outdir, 'gender_claim_table.csv'))
    # Ensure exactly two genders captured; if more, pick two main categories (Male/Female)
    if set(['M','F']).issubset(set(df['Gender'].unique())):
        # successes and nobs
        success = [gender_tab.loc['M',1], gender_tab.loc['F',1]]
        nobs = [gender_tab.loc['M',:].sum(), gender_tab.loc['F',:].sum()]
        stat, pval = two_proportion_ztest(success, nobs)
        results['gender_freq_ztest'] = dict(z=stat, p=pval, success=success, nobs=nobs)
        print(f"Gender z={stat:.3f}, p={pval:.6f}")
    else:
        # fallback: chi2 across all genders captured
        chi2, p, dof, _ = chi2_test(df, 'Gender')
        results['gender_freq_chi2'] = dict(chi2=chi2, p=p, dof=dof)
        print(f"Gender Chi2={chi2:.3f}, p={p:.6f}")

    # Severity by gender (claims only)
    claims_gender = claims.groupby('Gender')['TotalClaims'].agg(['count','mean','std'])
    claims_gender.to_csv(os.path.join(outdir, 'claims_by_gender.csv'))
    # If two main genders, perform t-test or Mann-Whitney
    genders_present = [g for g in df['Gender'].unique() if g in claims['Gender'].unique()]
    if len(genders_present) >= 2:
        grp_vals = [claims[claims['Gender'] == g]['TotalClaims'].values for g in genders_present]
        # use Mann-Whitney U (nonparametric)
        if len(grp_vals[0])>10 and len(grp_vals[1])>10:
            u, p = stats.mannwhitneyu(grp_vals[0], grp_vals[1], alternative='two-sided')
            results['gender_severity_mwu'] = dict(U=u, p=p, genders=genders_present)
            print(f"Mann-Whitney U={u:.3f}, p={p:.6f}")
        else:
            print("Insufficient sample size for robust gender severity test.")
    else:
        print("Not enough gender groups with claims to compare severity.")

    # Save results summary
    pd.Series({k: str(v) for k,v in results.items()}).to_csv(os.path.join(outdir, 'stat_test_results_summary.csv'), header=False)
    print("Saved results to:", outdir)
    return results

# Notebook-friendly
data_path = "../data/MachineLearningRating_v3.txt"
outdir = "../reports/task3"
os.makedirs(outdir, exist_ok=True)

df = load_data(data_path)
results = run_tests(df, outdir)

df = load_data(data_path)
results = run_tests(df, outdir)
    # print brief interpretation suggestions
print("\n--- Quick interpretation guide ---")
if results['province_freq_chi2']['p'] < 0.05:
        print("Reject H0: provinces differ in claim frequency.")
else:
        print("Fail to reject H0: no strong evidence provinces differ in claim frequency.")

C:\Users\mubar\AppData\Local\Temp\ipykernel_2468\2885341563.py:2: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep='|')


Testing H0: No risk differences across provinces (frequency)
Chi2=104.191, p=0.000000, dof=8
Kruskal H=106.093, p=0.000000
Testing top 10 PostalCodes: [2000, 122, 7784, 299, 7405, 458, 8000, 2196, 470, 7100]
Zip Chi2 (top10)=72.649, p=0.000000
Zip Kruskal H (top10)=41.428, p=0.000004
Zip Margin Kruskal (top10) H=4931.140, p=0.000000
Testing gender differences (frequency)
Gender Chi2=7.256, p=0.026570
Mann-Whitney U=138381.500, p=0.084725
Saved results to: ../reports/task3


C:\Users\mubar\AppData\Local\Temp\ipykernel_2468\2885341563.py:2: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep='|')


Testing H0: No risk differences across provinces (frequency)
Chi2=104.191, p=0.000000, dof=8
Kruskal H=106.093, p=0.000000
Testing top 10 PostalCodes: [2000, 122, 7784, 299, 7405, 458, 8000, 2196, 470, 7100]
Zip Chi2 (top10)=72.649, p=0.000000
Zip Kruskal H (top10)=41.428, p=0.000004
Zip Margin Kruskal (top10) H=4931.140, p=0.000000
Testing gender differences (frequency)
Gender Chi2=7.256, p=0.026570
Mann-Whitney U=138381.500, p=0.084725
Saved results to: ../reports/task3

--- Quick interpretation guide ---
Reject H0: provinces differ in claim frequency.
